## - Generate the 70/30 sampling from all CCEs

In [12]:
import pandas as pd
import random
from pathlib import Path

# =========================
# CONFIG (V3.1-aligned)
# =========================
INPUT_DIR  = Path(r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\combined_V3.0")
INPUT_CSV  = INPUT_DIR / "episodes_enriched_combined.csv"   # list of all CCEs (V3.1)

OUTPUT_DIR = Path(r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\Second_Label")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

#SAMPLE_SIZE        = 1000    # total size across DEV+TEST - Original Size
SAMPLE_SIZE        = 300    # total size across DEV+TEST
#SEED               = 12345   # fixed seed for reproducibility - Original Seed
SEED               = 1234567   # fixed seed for reproducibility
STRATIFY_BY_REPO   = False   # True = proportional per repo
#DEV_RATIO          = 0.70    # 70% to DEV, 30% to TEST - Original Ratio
DEV_RATIO          = 0.0    # 0% to DEV, 100% to TEST

# =========================
# HELPERS
# =========================
def sample_rows(df_in: pd.DataFrame, n: int, seed: int, stratify: bool) -> pd.DataFrame:
    """Sample n rows from df_in (optionally stratified by repo)."""
    if df_in.empty:
        return df_in
    n = min(n, len(df_in))
    if not stratify or "repo" not in df_in.columns:
        return df_in.sample(n=n, random_state=seed)

    # Proportional by repo
    rng = random.Random(seed)
    parts = []
    total = len(df_in)
    for repo, g in df_in.groupby("repo", sort=False):
        k = max(1, round(n * (len(g) / total)))
        parts.append(g.sample(n=min(k, len(g)), random_state=rng.randint(0, 10**9)))
    out = pd.concat(parts, ignore_index=True)
    if len(out) > n:
        out = out.sample(n=n, random_state=seed)  # trim to exact n
    return out

# =========================
# MAIN
# =========================
def main():
    # 1) Load all CCEs
    if not INPUT_CSV.exists():
        raise FileNotFoundError(f"Input file not found: {INPUT_CSV}")
    df = pd.read_csv(INPUT_CSV, encoding="utf-8-sig")
    if df.empty:
        raise ValueError("Input CCE list is empty.")
    print(f"[INFO] Loaded CCEs: {len(df):,} rows from {INPUT_CSV}")

    # Preserve original column order to enforce identical schema
    original_cols = df.columns.tolist()

    # 2) Draw overall sample of CCE rows (no column changes)
    sample_all = sample_rows(df, SAMPLE_SIZE, SEED, STRATIFY_BY_REPO)

    # 3) Split into DEV/TEST (70/30) with fixed seed
    sample_all = sample_all.sample(frac=1.0, random_state=SEED).reset_index(drop=True)  # shuffle once
    n_total = len(sample_all)
    n_dev = int(round(n_total * DEV_RATIO))
    n_test = n_total - n_dev

    dev_df  = sample_all.iloc[:n_dev].copy().reindex(columns=original_cols)
    test_df = sample_all.iloc[n_dev:].copy().reindex(columns=original_cols)

    # 4) Save full columns, identical to input (order preserved)
    dev_path  = OUTPUT_DIR / "DEV_Commit_Sample.csv"
    test_path = OUTPUT_DIR / "TEST_Commit_Sample.csv"

    dev_df.to_csv(dev_path, index=False, encoding="utf-8")
    test_df.to_csv(test_path, index=False, encoding="utf-8")

    # Optional sanity checks
    assert dev_df.columns.tolist()  == original_cols, "DEV columns differ from input!"
    assert test_df.columns.tolist() == original_cols, "TEST columns differ from input!"

    print(f"[OK] DEV written : {dev_path}  (rows={len(dev_df):,}, cols={len(original_cols)})")
    print(f"[OK] TEST written: {test_path} (rows={len(test_df):,}, cols={len(original_cols)})")
    print(f"[OK] Total sampled: {n_total:,} (DEV={len(dev_df):,}, TEST={len(test_df):,})")

if __name__ == "__main__":
    main()


[INFO] Loaded CCEs: 12,700 rows from C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\combined_V3.0\episodes_enriched_combined.csv
[OK] DEV written : C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\Second_Label\DEV_Commit_Sample.csv  (rows=0, cols=37)
[OK] TEST written: C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\Second_Label\TEST_Commit_Sample.csv (rows=300, cols=37)
[OK] Total sampled: 300 (DEV=0, TEST=300)
